# gesn-search: генерация данных, дообучение и оценка

Весь пайплайн на одной GPU-сессии Kaggle:
1. открытая LLM (Qwen2.5-7B-Instruct) пишет по 5 запросов к каждой норме ГЭСН;
2. запросы чистятся и превращаются в тройки «запрос — норма — трудный негатив»;
3. `multilingual-e5-base` дообучается на тройках;
4. все подходы считаются на ручном тесте (150 запросов) и синтетическом dev.

**Перед запуском** (справа, Session options): Accelerator — **GPU T4 x2**, Internet — **On**.
Затем **Run All**. Полный прогон — примерно 1,5–2 часа; вкладку можно закрыть, если запустить через **Save Version → Save & Run All**.

Результаты в Output: `results_test.md`, `results_dev.md`, `train.jsonl`, `dev.jsonl`, `synth_raw.jsonl`, `e5-gesn.zip`.

In [ ]:
# True — быстрый прогон на 30 нормах (~15 мин), чтобы проверить, что всё работает
SMOKE = False
LIMIT = "--limit 30" if SMOKE else ""

In [ ]:
!rm -rf /tmp/gesn-search && git clone -q https://github.com/malma0/gesn-search /tmp/gesn-search
%cd /tmp/gesn-search
!pip install -q -e . --no-deps
!pip install -q "rank-bm25>=0.2.2" "pymorphy3>=2.0" "sentence-transformers>=5.0" "accelerate>=1.1.0" datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Генерация запросов

In [ ]:
!python -m gesn.generate {LIMIT}

## 2. Очистка и тройки

In [ ]:
!python -m gesn.synth
!head -c 1500 data/train/train.jsonl

## 3. Дообучение

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.train

## 4. Оценка

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.evaluate --methods bm25 e5 hybrid ft=models/e5-gesn bm25+ft --out results_test.md

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.evaluate --qrels data/train/dev.jsonl --methods bm25 e5 ft=models/e5-gesn --out results_dev.md

## 5. Сохранение результатов

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working")
for f in ["results_test.md", "results_dev.md", "data/train/train.jsonl", "data/train/dev.jsonl", "data/train/synth_raw.jsonl"]:
    shutil.copy(f, out / Path(f).name)
shutil.make_archive(str(out / "e5-gesn"), "zip", "models/e5-gesn")
print(sorted(p.name for p in out.iterdir()))